# Hands-on — Recommendations API

Demonstração interativa das rotas da API de personalização.

**Setup automático:** a primeira célula de código resolve `API_BASE_URL` e `API_KEY` via `terraform output` / SSM (mesmo fluxo de `testing_endpoint.ipynb`). Basta ter infra deployada e credenciais AWS configuradas na máquina.

Variáveis opcionais (sobrescrevem o auto-resolve):

```bash
export RECOMMENDATIONS_API_BASE_URL="https://<api-id>.execute-api.us-east-1.amazonaws.com/v1"
export RECOMMENDATIONS_API_KEY="<sua-api-key>"
export RECOMMENDATIONS_TEST_USER_ID="u_0231"
export RECOMMENDATIONS_TEST_COLD_START_USER_ID="u_9999"
```

## Setup — imports, URL e API key

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import httpx
import pandas as pd

PROJECT_ROOT = (Path("..") if Path("..").joinpath("terraform").is_dir() else Path(".")).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
TERRAFORM_DIR = PROJECT_ROOT / "terraform"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_STAGE = "v1"
KNOWN_USER_ID = os.getenv("RECOMMENDATIONS_TEST_USER_ID", "u_0231")
COLD_START_USER_ID = os.getenv("RECOMMENDATIONS_TEST_COLD_START_USER_ID", "u_9999")


def _terraform_output(name: str) -> str | None:
    try:
        return subprocess.check_output(
            ["terraform", f"-chdir={TERRAFORM_DIR}", "output", "-raw", name],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def normalize_api_base_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith(f"/{API_STAGE}"):
        return normalized
    if "execute-api" in normalized:
        return f"{normalized}/{API_STAGE}"
    return normalized


def load_api_config() -> tuple[str, str]:
    base_url = os.getenv("RECOMMENDATIONS_API_BASE_URL")
    api_key = os.getenv("RECOMMENDATIONS_API_KEY")

    if not base_url:
        base_url = _terraform_output("recommendations_api_gateway_endpoint")

    if not api_key:
        api_key = _terraform_output("recommendations_api_key")

    if not api_key:
        param_name = _terraform_output("recommendations_api_key_ssm_parameter")
        if param_name:
            import boto3

            ssm = boto3.client("ssm", region_name=AWS_REGION)
            api_key = ssm.get_parameter(Name=param_name, WithDecryption=True)[
                "Parameter"
            ]["Value"]

    if not base_url or not api_key:
        raise RuntimeError(
            "Defina RECOMMENDATIONS_API_BASE_URL e RECOMMENDATIONS_API_KEY "
            "ou aplique o Terraform e configure credenciais AWS."
        )

    return normalize_api_base_url(base_url), api_key


def api_headers(*, with_key: bool = True) -> dict[str, str]:
    headers = {"Accept": "application/json"}
    if with_key:
        headers["x-api-key"] = API_KEY
    return headers


def call_api(
    method: str,
    path: str,
    *,
    json_body: dict | None = None,
    with_key: bool = True,
    timeout: float = 30.0,
) -> httpx.Response:
    url = f"{API_BASE_URL}{path}"
    with httpx.Client(timeout=timeout) as client:
        return client.request(
            method,
            url,
            headers=api_headers(with_key=with_key),
            json=json_body,
        )


def show_response(response: httpx.Response, *, label: str = "") -> httpx.Response:
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}{response.request.method} {response.request.url}")
    print(f"{prefix}HTTP {response.status_code}")
    content_type = response.headers.get("content-type", "")
    if "json" in content_type:
        print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    else:
        print(response.text)
    return response


def wait_for_api_health(
    *,
    timeout_seconds: float = 300,
    poll_interval_seconds: float = 10,
) -> None:
    deadline = time.monotonic() + timeout_seconds
    last_status: int | str = "unknown"
    while time.monotonic() < deadline:
        try:
            response = call_api("GET", "/health", with_key=False, timeout=10.0)
            last_status = response.status_code
            if response.status_code == 200:
                return
        except httpx.HTTPError:
            last_status = "connection_error"
        time.sleep(poll_interval_seconds)
    raise TimeoutError(
        f"/health did not return 200 within {timeout_seconds}s (last_status={last_status})"
    )


def warn_if_predictions_table_empty() -> None:
    try:
        from tests.helpers.aws_integration import (
            dynamodb_table_has_items,
            load_terraform_outputs,
        )

        outputs = load_terraform_outputs()
        table_name = outputs.get("predictions_dynamodb_table_name")
        if table_name and not dynamodb_table_has_items(table_name):
            print(
                f"AVISO: tabela DynamoDB '{table_name}' vazia. "
                "Rode model_predict antes dos testes de usuário conhecido."
            )
    except Exception as error:  # noqa: BLE001
        print(f"AVISO: não foi possível verificar DynamoDB ({error}).")


API_BASE_URL, API_KEY = load_api_config()
wait_for_api_health()
warn_if_predictions_table_empty()

print(f"API base URL: {API_BASE_URL}")
print(f"Known user:   {KNOWN_USER_ID}")
print(f"Cold start:   {COLD_START_USER_ID}")

## Leitura de dados necessários

In [ ]:
events = pd.read_csv(DATA_DIR / "events.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

print(f"events:   {len(events):,} linhas · {events['user_id'].nunique()} usuários")
print(f"products: {len(products):,} produtos · categorias: {sorted(products['category'].unique())}")

sample_user = KNOWN_USER_ID
user_events = events[events["user_id"] == sample_user]
print(f"\nUsuário demo {sample_user}: {len(user_events)} eventos")
display(user_events.head(3))

## Requisições por rota

### `GET /health`

In [ ]:
show_response(call_api("GET", "/health", with_key=False), label="health")

### `GET /recommendations/{user_id}`

In [ ]:
print("--- usuário com histórico ---")
show_response(
    call_api("GET", f"/recommendations/{KNOWN_USER_ID}"),
    label="recommendations",
)

print("\n--- cold start ---")
show_response(
    call_api("GET", f"/recommendations/{COLD_START_USER_ID}"),
    label="cold_start",
)

### `GET /metrics` — Prometheus (default)

In [ ]:
show_response(call_api("GET", "/metrics"), label="metrics_prometheus")

### `GET /metrics?format=datadog`

In [ ]:
show_response(call_api("GET", "/metrics?format=datadog"), label="metrics_datadog")

### `POST /recommendations_filtered` — exemplo básico

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={"user_id": KNOWN_USER_ID, "limit": 5},
    ),
    label="filtered_basic",
)

### `POST /recommendations_filtered` — todos os filtros

Cada célula abaixo demonstra um filtro (ou par de filtros) do contrato da API.

In [ ]:
baseline = call_api("GET", f"/recommendations/{KNOWN_USER_ID}").json()
sample_product_id = baseline["recommendations"][0]["product_id"]
sample_category = products.loc[products["product_id"] == sample_product_id, "category"].iloc[0]
price_stats = products["price"]
min_price_demo = float(price_stats.quantile(0.25))
max_price_demo = float(price_stats.quantile(0.75))

print(f"Produto de referência: {sample_product_id} · categoria: {sample_category}")
print(f"Faixa de preço demo: {min_price_demo:.2f} – {max_price_demo:.2f}")

#### `limit`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={"user_id": KNOWN_USER_ID, "limit": 3},
    ),
    label="filter_limit",
)

#### `exclude_product_ids`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_product_ids": [sample_product_id],
        },
    ),
    label="filter_exclude_product_ids",
)

#### `category`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "category": sample_category,
        },
    ),
    label="filter_category",
)

#### `categories` (whitelist com várias categorias)

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "categories": ["esporte", "moda"],
        },
    ),
    label="filter_categories",
)

#### `exclude_categories`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_categories": ["livros"],
        },
    ),
    label="filter_exclude_categories",
)

#### `min_price` / `max_price`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_price": min_price_demo,
            "max_price": max_price_demo,
        },
    ),
    label="filter_price_range",
)

#### `min_avg_rating`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_avg_rating": 4.0,
        },
    ),
    label="filter_min_avg_rating",
)

#### `min_popularity_score`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_popularity_score": 0.3,
        },
    ),
    label="filter_min_popularity_score",
)

#### `min_recommendation_score`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "min_recommendation_score": 0.1,
        },
    ),
    label="filter_min_recommendation_score",
)

#### `only_affinity_match`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "only_affinity_match": True,
        },
    ),
    label="filter_only_affinity_match",
)

#### `exclude_cold_start` (usuário cold start para comparar)

In [ ]:
print("--- com cold start (default) ---")
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={"user_id": COLD_START_USER_ID, "limit": 3},
    ),
    label="cold_start_with_fallback",
)

print("\n--- exclude_cold_start=true (pode retornar vazio) ---")
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": COLD_START_USER_ID,
            "limit": 3,
            "exclude_cold_start": True,
        },
    ),
    label="filter_exclude_cold_start",
)

#### Combinação de filtros

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_product_ids": [sample_product_id],
            "category": "esporte",
            "min_price": 50,
            "max_price": 500,
            "min_recommendation_score": 0.05,
            "only_affinity_match": True,
        },
    ),
    label="filter_combined",
)